<h1 align="center">Homework 2</h1>

**Assigned:** Session 3, October 10, 2026

**Due:** Session 5, October 24, 2026

**Covers:** Lectures 3 and 4

**Instructions:** This set is a single Jupyter notebook, distributed and submitted as a `.ipynb` file; nothing is submitted on paper. Each problem is a markdown cell followed by one or more code cells with a marked location for your work; do not change the markdown, the function signatures, or the final self-check cell. Use only the Python standard library and NumPy. Run every cell from a clean kernel before submitting, top to bottom, and submit the notebook with its output retained rather than cleared, so that a reader can see the results without re-executing anything. Part A can be attempted after Lecture 3 and Part B after Lecture 4.

**Conventions:** Qubits are labelled left to right, so in a state vector for $\ket{ab}$ the first qubit is $a$; the two-qubit basis is ordered $\ket{00}, \ket{01}, \ket{10}, \ket{11}$; a gate applied first is written rightmost in a matrix product. Basis states, gates, and the two-qubit basis order match Homeworks 1 and 3.

## Setup

Run this cell first. It defines the single-qubit states and gates used throughout the set, and one helper, `kron`, for building a multi-qubit operator or state as a tensor product. Do not modify this cell.

In [ ]:
import numpy as np

k0 = np.array([1, 0], dtype=complex)
k1 = np.array([0, 1], dtype=complex)
plus = (k0 + k1) / np.sqrt(2)
minus = (k0 - k1) / np.sqrt(2)

I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
CNOT = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], dtype=complex)


def kron(*factors):
    """Tensor product of two or more single-qubit (or larger) operators or state vectors, in the order given."""
    out = factors[0]
    for factor in factors[1:]:
        out = np.kron(out, factor)
    return out


def ket(bits):
    """Computational basis state from a bit string, e.g. ket('10') = |1> tensor |0>."""
    return kron(*[(k1 if b == "1" else k0) for b in bits])


def close(a, b, tol=1e-9):
    return np.allclose(a, b, atol=tol)

## Part A: circuits and teleportation

Attempt after Lecture 3.

### Problem A1: order matters

Consider two qubits with input $\ket{10}$. Circuit 1 applies $\mathrm{CNOT}$ (first qubit as control), then $H$ on the first qubit and $X$ on the second. Circuit 2 applies $H$ on the first qubit and $X$ on the second, then $\mathrm{CNOT}$.

Complete `circuit_1` and `circuit_2` below so each returns the four-entry output state vector as a NumPy array, built as an explicit matrix product of the gates listed above applied to `ket("10")`. Do not simplify the product by hand first; compose the matrices in code, in the order the two circuits apply them, exactly as you would write them applied to a state right to left.

In [ ]:
def circuit_1():
    # CNOT first, then H on qubit 1 and X on qubit 2.
    # TODO: build the two-qubit operator (H tensor X) @ CNOT and apply it to ket("10").
    raise NotImplementedError


def circuit_2():
    # H on qubit 1 and X on qubit 2 first, then CNOT.
    # TODO: build the two-qubit operator CNOT @ (H tensor X) and apply it to ket("10").
    raise NotImplementedError

Self-check. Circuit 1 should give the product state $\ket{-} \otimes \ket{0}$; Circuit 2 should give the entangled singlet $\Psi^- = (\ket{01} - \ket{10})/\sqrt{2}$. Do not edit this cell.

In [ ]:
out1 = circuit_1()
out2 = circuit_2()
assert close(out1, kron(minus, k0)), "circuit_1 should equal |-> tensor |0>"
assert close(
    out2, (ket("01") - ket("10")) / np.sqrt(2)
), "circuit_2 should equal the singlet Psi-"
print("A1 passed.")

### Problem A2: the teleportation protocol

Alice holds qubit 1 in an unknown state $\ket{\psi} = \alpha\ket{0} + \beta\ket{1}$ and qubit 2; Bob holds qubit 3; qubits 2 and 3 start in $\Phi^+ = (\ket{00} + \ket{11})/\sqrt{2}$. Alice applies $\mathrm{CNOT}$ from qubit 1 to qubit 2, then $H$ to qubit 1, and measures both of her qubits, obtaining bits $(m_1, m_2)$. The lecture showed that Bob's qubit is then $Z^{m_1}X^{m_2}\ket{\psi}$ up to a global phase, and that he recovers $\ket{\psi}$ by applying $X^{m_2}Z^{m_1}$.

Complete `after_alice(psi)` so it returns the three-qubit state after Alice's two gates, for an arbitrary single-qubit state `psi` (a length-2 NumPy array). Complete `bob_branch(state3, m1, m2)` so it extracts Bob's (unnormalized) one-qubit state conditional on Alice's outcome $(m_1, m_2)$, and `recover(bob_state, m1, m2)` so it applies the correction and returns a normalized state.

In [ ]:
def after_alice(psi):
    # psi is Alice's unknown qubit. Build psi (x) Phi+, apply CNOT on qubits 1,2, then H on qubit 1.
    # TODO
    raise NotImplementedError


def bob_branch(state3, m1, m2):
    # state3 is the 8-entry state from after_alice. Extract the length-2 slice of Bob's qubit (qubit 3)
    # conditional on Alice's two qubits reading (m1, m2), reshaping as needed.
    # TODO
    raise NotImplementedError


def recover(bob_state, m1, m2):
    # Apply the correction X^m2 Z^m1 and return a normalized state.
    # TODO
    raise NotImplementedError

Self-check. For several random single-qubit states, every one of Alice's four outcomes should recover the original state exactly (up to a global phase, which the check below removes by comparing $|\braket{\psi}{\psi'}|$ to 1) and each of Bob's four branches should carry probability $1/4$. Do not edit this cell.

In [ ]:
rng = np.random.default_rng(0)
for _ in range(20):
    a, b = rng.normal(size=2) + 1j * rng.normal(size=2)
    psi = np.array([a, b])
    psi /= np.linalg.norm(psi)
    state3 = after_alice(psi)
    for m1 in (0, 1):
        for m2 in (0, 1):
            branch = bob_branch(state3, m1, m2)
            assert close(
                np.linalg.norm(branch) ** 2, 0.25
            ), "each branch should carry probability 1/4"
            fixed = recover(branch, m1, m2)
            assert (
                abs(abs(np.vdot(psi, fixed)) - 1) < 1e-9
            ), f"recovery failed for outcome ({m1},{m2})"
print("A2 passed.")

### Problem A3: no-cloning, computationally

Problem A1 of Homework 1 shows by hand that no unitary $U$ can satisfy $U \ket{00} = \ket{00}$, $U \ket{10} = \ket{11}$, and $U \ket{+} \ket{0} = \ket{+} \ket{+}$ simultaneously, because linearity forces $U \ket{+} \ket{0}$ to equal $\Phi^+$, not $\ket{+} \ket{+}$.

Complete `classical_copy()` to return the $4 \times 4$ matrix of the gate that correctly copies $\ket{0}$ and $\ket{1}$ (a controlled-NOT with the first qubit as control and the second prepared in $\ket{0}$), and `would_be_cloner_output()` to return what that same gate produces when the first qubit is $\ket{+}$ instead. The self-check below verifies the gate copies the two classical states but does not produce $\ket{+} \ket{+}$, confirming computationally that a device copying $\ket{0}$ and $\ket{1}$ is not a universal cloner.

In [ ]:
def classical_copy():
    # TODO: return the CNOT matrix (already defined above; this problem asks you to name it in context).
    raise NotImplementedError


def would_be_cloner_output():
    # TODO: apply classical_copy() to |+>|0> and return the resulting state.
    raise NotImplementedError

Self-check. Do not edit this cell.

In [ ]:
U = classical_copy()
assert close(U @ ket("00"), ket("00")), "should copy |0>"
assert close(U @ ket("10"), ket("11")), "should copy |1>"
out = would_be_cloner_output()
phi_plus = (ket("00") + ket("11")) / np.sqrt(2)
plus_plus = kron(plus, plus)
assert close(out, phi_plus), "CNOT on |+>|0> should give Phi+, not |+>|+>"
assert not close(out, plus_plus), "a genuine cloner would have produced |+>|+>"
print("A3 passed.")

## Part B: interference and the Deutsch-Jozsa algorithm

Attempt after Lecture 4.

### Problem B1: building an oracle from a Python function

For an $n$-bit function $f$, the oracle acts as $U_f\ket{x}\ket{y} = \ket{x}\ket{y \oplus f(x)}$. Complete `oracle_matrix(n, f)` so that, given `n` and a Python function `f` mapping an integer in $\{0, \ldots, 2^n - 1\}$ to a bit, it returns the $2^{n+1} \times 2^{n+1}$ permutation matrix of $U_f$, with the $n$ input qubits on top and the one target qubit on the bottom wire, matching the convention above.

In [ ]:
def oracle_matrix(n, f):
    # TODO: build the (2**(n+1)) x (2**(n+1)) permutation matrix for U_f.
    # Basis index for |x>|y> is x * 2 + y.
    raise NotImplementedError

Self-check: $U_f$ must be a permutation matrix (unitary, with exactly one 1 per row and column), and applying it twice must return the identity, since $y \oplus f(x) \oplus f(x) = y$. Do not edit this cell.

In [ ]:
f_xor = lambda x: (x >> 1) ^ (x & 1)
U = oracle_matrix(2, f_xor)
assert close(U @ U.conj().T, np.eye(8)), "U_f must be unitary"
assert close(U @ U, np.eye(8)), "applying U_f twice must be the identity"
print("B1 passed.")

### Problem B2: the Deutsch-Jozsa circuit

Complete `deutsch_jozsa(n, f)` so it runs the full algorithm: prepare the $n$-qubit input register in $\ket{0}^{\otimes n}$ and the target in $\ket{-}$, apply $H^{\otimes n}$ to the input register, apply $U_f$, apply $H^{\otimes n}$ to the input register again, and return the resulting state of the input register alone (trace out the target by using the fact that it factors out as $\ket{-}$ throughout, since the target is never measured).

Run it on $f(x_1, x_2) = x_1 \oplus x_2$, on $g(x_1, x_2) = x_1$, and on the constant function $h(x) = 1$, and compare the outputs to the values computed by hand in Homework 1.

In [ ]:
def deutsch_jozsa(n, f):
    # TODO: implement the circuit described above and return the n-qubit input register state
    # (a vector of length 2**n).
    raise NotImplementedError

Self-check. Do not edit this cell.

In [ ]:
g_x1 = lambda x: x >> 1
h_one = lambda x: 1
out_xor = deutsch_jozsa(2, f_xor)
out_g = deutsch_jozsa(2, g_x1)
out_h = deutsch_jozsa(2, h_one)
assert close(
    np.abs(out_xor), np.eye(4)[3]
), "balanced x1 xor x2 should give |11> with certainty"
assert close(np.abs(out_g), np.eye(4)[2]), "balanced x1 should give |10> with certainty"
assert close(out_h, -np.eye(4)[0]), "constant function should give -|00>"
print("B2 passed.")

### Problem B3: why the promise matters

Let $k(00) = k(01) = k(10) = 0$ and $k(11) = 1$, a function that is neither constant nor balanced. Use `deutsch_jozsa` to compute its output amplitudes, and verify that all four outcomes are equally likely.

In [ ]:
def k_neither(x):
    # TODO: return 0 for x in {0, 1, 2} and 1 for x == 3 (the function k above).
    raise NotImplementedError

Self-check: the amplitudes should be $\tfrac{1}{2}, \tfrac{1}{2}, \tfrac{1}{2}, -\tfrac{1}{2}$, and every outcome probability should be $\tfrac{1}{4}$. Do not edit this cell.

In [ ]:
out_k = deutsch_jozsa(2, k_neither)
assert close(
    out_k, np.array([0.5, 0.5, 0.5, -0.5])
), "amplitudes should be 1/2, 1/2, 1/2, -1/2"
assert close(
    np.abs(out_k) ** 2, 0.25 * np.ones(4)
), "every outcome should have probability 1/4"
print("B3 passed.")

### Problem B4: classical queries against a random balanced function

Lecture 4 shows that a randomized classical algorithm querying $k$ random inputs and answering "constant" when they all agree is wrong, on a balanced function, with probability at most $2^{1-k}$; for $k = 8$ this is below $1\%$.

Complete `random_query_trial(n, f, k, rng)` so it draws `k` independent uniform random inputs in $\{0, \ldots, 2^n - 1\}$ (using the supplied NumPy random generator `rng`), queries `f` at each, and returns `True` if the algorithm is fooled (all `k` values agree even though `f` is balanced). Then run enough trials at $n = 3$, $k = 8$, on a fixed balanced function to estimate the failure rate, and compare it to the bound $2^{1-k}$.

In [ ]:
def random_query_trial(n, f, k, rng):
    # TODO: sample k inputs uniformly at random with replacement, query f on each, and return True
    # exactly when all k values agree (the algorithm is fooled, since f is balanced by construction).
    raise NotImplementedError

Self-check: a balanced function on $n = 3$ bits, sampled 20000 times with $k = 8$ queries per trial, should be fooled at a rate close to the bound $2^{1-k} = 2^{-7} \approx 0.0078$, comfortably under $1\%$. The Monte Carlo estimate is allowed some statistical slack. Do not edit this cell.

In [ ]:
balanced_3 = lambda x: x & 1  # balanced on 3 bits: half the inputs give 0, half give 1
rng = np.random.default_rng(1)
trials = 20000
fooled = sum(random_query_trial(3, balanced_3, 8, rng) for _ in range(trials))
rate = fooled / trials
bound = 2 ** (1 - 8)
assert (
    rate < 0.02
), f"observed failure rate {rate:.4f} should be well under the 1% design target"
print(f"B4 passed: observed failure rate {rate:.4f} against bound {bound:.4f}.")

## Summary

Parts A and B are complete once every self-check cell above prints `passed`. Submission arrangements are announced separately.